In [2]:
import duckdb
con = duckdb.connect("../olist.duckdb")

In [3]:
con.execute("""
SELECT
    COUNT(*)                                          AS total_orders,
    COUNT(*) - COUNT(order_approved_at)               AS null_approved,
    COUNT(*) - COUNT(order_delivered_carrier_date)    AS null_carrier,
    COUNT(*) - COUNT(order_delivered_customer_date)   AS null_delivered,
    COUNT(*) - COUNT(order_estimated_delivery_date)   AS null_estimated
FROM raw_orders
""").df()

,total_orders,null_approved,null_carrier,null_delivered,null_estimated
0,99441,160,1783,2965,0


In [4]:
con.execute("""
SELECT 'orders' AS tbl, COUNT(*) AS duplicate_keys FROM (
    SELECT order_id FROM raw_orders GROUP BY order_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'customers', COUNT(*) FROM (
    SELECT customer_id FROM raw_customers GROUP BY customer_id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'reviews', COUNT(*) FROM (
    SELECT review_id FROM raw_reviews GROUP BY review_id HAVING COUNT(*) > 1)
""").df()

,tbl,duplicate_keys
0,orders,0
1,customers,0
2,reviews,789


In [5]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM raw_orders o
     LEFT JOIN raw_customers c ON o.customer_id = c.customer_id
     WHERE c.customer_id IS NULL)                 AS orders_without_customer,
    (SELECT COUNT(*) FROM raw_order_items i
     LEFT JOIN raw_orders o ON i.order_id = o.order_id
     WHERE o.order_id IS NULL)                    AS items_without_order,
    (SELECT COUNT(*) FROM raw_orders o
     LEFT JOIN raw_order_items i ON o.order_id = i.order_id
     WHERE i.order_id IS NULL)                    AS orders_without_items
""").df()

,orders_without_customer,items_without_order,orders_without_items
0,0,0,775


In [6]:
con.execute("""
SELECT
    SUM(CASE WHEN order_delivered_customer_date < order_purchase_timestamp
             THEN 1 ELSE 0 END) AS delivered_before_purchase,
    SUM(CASE WHEN order_approved_at < order_purchase_timestamp
             THEN 1 ELSE 0 END) AS approved_before_purchase,
    SUM(CASE WHEN order_delivered_customer_date < order_delivered_carrier_date
             THEN 1 ELSE 0 END) AS customer_before_carrier,
    SUM(CASE WHEN order_status = 'delivered'
              AND order_delivered_customer_date IS NULL
             THEN 1 ELSE 0 END) AS delivered_but_no_date
FROM raw_orders
""").df()

,delivered_before_purchase,approved_before_purchase,customer_before_carrier,delivered_but_no_date
0,0.0,0.0,23.0,8.0


In [7]:
con.execute("""
SELECT order_status,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM raw_orders
GROUP BY order_status
ORDER BY orders DESC
""").df()

,order_status,orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [8]:
con.execute("""
SELECT date_trunc('month', order_purchase_timestamp) AS month,
       COUNT(*) AS orders
FROM raw_orders
GROUP BY 1 ORDER BY 1
""").df()

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [9]:
con.execute("""
SELECT COUNT(*)                            AS rows,
       COUNT(DISTINCT customer_id)         AS distinct_customer_id,
       COUNT(DISTINCT customer_unique_id)  AS distinct_unique_id
FROM raw_customers
""").df()

,rows,distinct_customer_id,distinct_unique_id
0,99441,99441,96096


In [10]:
con.execute("""
SELECT o.order_status,
       COUNT(*) AS orders_with_no_items
FROM raw_orders o
LEFT JOIN raw_order_items i ON o.order_id = i.order_id
WHERE i.order_id IS NULL
GROUP BY o.order_status
ORDER BY orders_with_no_items DESC
""").df()

,order_status,orders_with_no_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [11]:
con.execute("""
SELECT sellers_on_order, COUNT(*) AS orders
FROM (
    SELECT order_id, COUNT(DISTINCT seller_id) AS sellers_on_order
    FROM raw_order_items GROUP BY order_id
)
GROUP BY 1 ORDER BY 1
""").df()

,sellers_on_order,orders
0,1,97388
1,2,1219
2,3,54
3,4,3
4,5,2


In [12]:
con.execute("""
SELECT reviews_per_order, COUNT(*) AS orders
FROM (
    SELECT order_id, COUNT(*) AS reviews_per_order
    FROM raw_reviews GROUP BY order_id
)
GROUP BY 1 ORDER BY 1
""").df()

,reviews_per_order,orders
0,1,98126
1,2,543
2,3,4


In [13]:
con.execute("""
SELECT
    COUNT(*) AS orders_with_multiple_reviews,
    SUM(CASE WHEN min_score = max_score THEN 1 ELSE 0 END) AS same_score,
    SUM(CASE WHEN min_score <> max_score THEN 1 ELSE 0 END) AS different_score
FROM (
    SELECT order_id,
           MIN(review_score) AS min_score,
           MAX(review_score) AS max_score
    FROM raw_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
""").df()

,orders_with_multiple_reviews,same_score,different_score
0,547,345.0,202.0
